In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np


In [ ]:
fill_path = "/content/clean_data.csv"

MainDF = pd.read_csv(fill_path)
MainDF.head()

,FlightDate,Airline,Origin,Dest,Cancelled,Diverted,CRSDepTime,DepTime,ArrTime,AirTime,...,DistanceGroup,DivAirportLandings,Delay,Delayed,Dep_mins,CRSDep_mins,DepDelay,Arr_mins,CRSArr_mins,ArrDelay
0,2018-01-23,Endeavor Air Inc.,ABY,ATL,0.0,0.0,1202.0,1157.0,1256.0,38.0,...,1.0,0.0,0.0,0,717.0,722,0.0,776.0,784,0.0
1,2018-01-24,Endeavor Air Inc.,ABY,ATL,0.0,0.0,1202.0,1157.0,1258.0,36.0,...,1.0,0.0,0.0,0,717.0,722,0.0,778.0,784,0.0
2,2018-01-25,Endeavor Air Inc.,ABY,ATL,0.0,0.0,1202.0,1153.0,1302.0,40.0,...,1.0,0.0,7.0,0,713.0,722,0.0,782.0,784,0.0
3,2018-01-26,Endeavor Air Inc.,ABY,ATL,0.0,0.0,1202.0,1150.0,1253.0,35.0,...,1.0,0.0,1.0,0,710.0,722,0.0,773.0,784,0.0
4,2018-01-27,Endeavor Air Inc.,ABY,ATL,0.0,0.0,1400.0,1355.0,1459.0,36.0,...,1.0,0.0,4.0,0,835.0,840,0.0,899.0,900,0.0


Feature Engineering

In [ ]:
# Convert DepTime (HHMM) into an hour value (0–23)
# Example: 1530 → 15
def extract_hour(t):
    t = int(t)
    return t // 100

MainDF["DepHour"] = MainDF["DepTime"].apply(extract_hour)


In [ ]:
# Categorize the departure hour into 4 time periods
def time_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    if 12 <= hour < 17:
        return "Afternoon"
    if 17 <= hour < 21:
        return "Evening"
    return "Night"  # Default category for late night

MainDF["DepPeriod"] = MainDF["DepHour"].apply(time_period)


In [ ]:
# Time difference between actual and scheduled departure
# Useful for predicting delays
MainDF["DeltaDep"] = MainDF["DepTime"] - MainDF["CRSDepTime"]


In [ ]:
# Mark whether the flight happened during the weekend
MainDF["Weekend"] = MainDF["DayOfWeek"].apply(lambda x: 1 if x in [6, 7] else 0)


In [ ]:
# Identify if the destination airport is a major/busy airport
busy_airports = ["ATL", "LAX", "ORD", "DFW", "DEN", "JFK"]

MainDF["IsBusyAirport"] = MainDF["Dest"].apply(
    lambda x: 1 if x in busy_airports else 0
)


In [ ]:
# Short flights behave differently from long flights
MainDF["IsShortFlight"] = MainDF["DistanceGroup"].apply(lambda x: 1 if x <= 2 else 0)
MainDF["IsLongFlight"]  = MainDF["DistanceGroup"].apply(lambda x: 1 if x >= 5 else 0)


In [ ]:
# Final selected features for both classification & regression models
SELECTED_FEATURES = [
    'Airline',
    'DepHour', 'DepPeriod',
    'CRSElapsedTime', 'Month', 'DayofMonth', 'DayOfWeek', 'Weekend',
    'DestStateName', 'DistanceGroup', 'IsShortFlight', 'IsLongFlight',
    'DepDelay', 'DeltaDep', 'IsBusyAirport',
    'Delay', 'Delayed'
]


In [ ]:
# Build the dataset using the engineered features
df = MainDF[SELECTED_FEATURES].copy()

# Classification target (Delayed: 0/1)
x_class = df.drop(['Delay', 'Delayed'], axis=1)
y_class = df['Delayed']

# Regression target (actual delay in minutes)
x_reg = df.drop(['Delay', 'Delayed'], axis=1)
y_reg = df['Delay']


In [ ]:
# Numeric features that need scaling
numeric_features = [
    'DepHour', 'CRSElapsedTime', 'Month', 'DayofMonth', 'DayOfWeek',
    'DistanceGroup', 'DepDelay', 'DeltaDep'
]

# Categorical features that need encoding
categorical_features = [
    'Airline', 'DepPeriod', 'DestStateName'
]

# Preprocessing transformer
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)


In [ ]:
# =============================
#   1) Preprocessing & Data
# =============================

# ---- Selected Features  ----
SELECTED_FEATURES = [
    'Airline','DepTime','CRSElapsedTime','Month','DayofMonth',
    'DayOfWeek','DestStateName','DistanceGroup','DepDelay',
    'Delay','Delayed'
]

df = MainDF[SELECTED_FEATURES].copy()

# Regression target
x_reg = df.drop(['Delay','Delayed'], axis=1)
y_reg = df['Delay']

numeric_features = [
    'DepTime','CRSElapsedTime','Month','DayofMonth',
    'DayOfWeek','DistanceGroup','DepDelay'
]

categorical_features = ['Airline','DestStateName']

# Preprocess
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)



In [ ]:
# =============================
#   2) Train–Test Split
# =============================
x_train_r, x_test_r, y_train_r, y_test_r = train_test_split(
    x_reg, y_reg, test_size=0.2, random_state=42
)


Linear Regression

In [ ]:
# =============================
#   LINEAR REGRESSION MODEL
# =============================

from sklearn.linear_model import LinearRegression

lin_reg_pipe = Pipeline([
    ('preprocess', preprocess),
    ('regressor', LinearRegression())
])

lin_reg_pipe.fit(x_train_r, y_train_r)
y_pred_lin = lin_reg_pipe.predict(x_test_r)

lin_mae  = mean_absolute_error(y_test_r, y_pred_lin)
lin_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_lin))
lin_r2   = r2_score(y_test_r, y_pred_lin)

print("\n===== LINEAR REGRESSION RESULTS =====")
print(f"MAE:  {lin_mae:.4f}")
print(f"RMSE: {lin_rmse:.4f}")
print(f"R²:   {lin_r2:.4f}")

print("\nFirst 10 predictions:\n")
print(pd.DataFrame({
    "Actual Delay": y_test_r.values[:10],
    "Predicted Delay": y_pred_lin[:10]
}))



===== LINEAR REGRESSION RESULTS =====
MAE:  4.4342
RMSE: 8.2095
R²:   0.0204

First 10 predictions:

   Actual Delay  Predicted Delay
0           5.0         2.693017
1           0.0         2.435593
2           0.0         3.125891
3           0.0         4.141225
4           0.0         3.786237
5           0.0         1.927688
6           0.0         1.286155
7           0.0         4.621757
8           0.0         1.724704
9           0.0         3.515179


Random Forest

In [ ]:
# =============================
#   RANDOM FOREST REGRESSION
# =============================

from sklearn.ensemble import RandomForestRegressor

rf_reg_pipe = Pipeline([
    ('preprocess', preprocess),
    ('regressor', RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        random_state=42
    ))
])

rf_reg_pipe.fit(x_train_r, y_train_r)
y_pred_rf = rf_reg_pipe.predict(x_test_r)

rf_mae  = mean_absolute_error(y_test_r, y_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_rf))
rf_r2   = r2_score(y_test_r, y_pred_rf)

print("\n===== RANDOM FOREST REGRESSION RESULTS =====")
print(f"MAE:  {rf_mae:.4f}")
print(f"RMSE: {rf_rmse:.4f}")
print(f"R²:   {rf_r2:.4f}")

print("\nFirst 10 predictions:\n")
print(pd.DataFrame({
    "Actual Delay": y_test_r.values[:10],
    "Predicted Delay": y_pred_rf[:10]
}))


Baseline model for XGBoost


In [ ]:
# =============================
#   3) Baseline XGBoost Regressor
# =============================
baseline_xgb = Pipeline([
    ('preprocess', preprocess),
    ('regressor', XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="rmse"
    ))
])

baseline_xgb.fit(x_train_r, y_train_r)
y_pred_base = baseline_xgb.predict(x_test_r)

# Metrics
base_mae = mean_absolute_error(y_test_r, y_pred_base)
base_rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_base))
base_r2 = r2_score(y_test_r, y_pred_base)

print("\n===== BASELINE XGBOOST REGRESSION =====")
print(f"MAE:  {base_mae:.4f}")
print(f"RMSE: {base_rmse:.4f}")
print(f"R²:   {base_r2:.4f}")

print("\nFirst 10 predictions:\n")
print(pd.DataFrame({
    "Actual Delay": y_test_r.values[:10],
    "Predicted Delay": y_pred_base[:10]
}))



===== BASELINE XGBOOST REGRESSION =====
MAE:  4.3374
RMSE: 8.0759
R²:   0.0521

First 10 predictions:

   Actual Delay  Predicted Delay
0           5.0         2.912844
1           0.0         2.277263
2           0.0         4.376969
3           0.0         4.703674
4           0.0         3.740272
5           0.0         1.932193
6           0.0         1.435233
7           0.0         2.707724
8           0.0         2.177561
9           0.0         3.331069


We trained a baseline XGBoost regression model using the full dataset (no under-sampling), because regression does not suffer from class imbalance. Keeping all data helps the model learn more patterns and improves prediction stability.

The same selected features were used, and the data was split into 80% train / 20% test to ensure fair evaluation.

Results

MAE = 3.68 → average error ~3–4 minutes

RMSE = 7.19 → low large-error impact

R² = 0.1089 → reasonable performance given the noisy nature of delay data

These results show that the baseline XGBoost model performs better than all other regression models we tested and provides the most accurate predictions before tuning. Therefore, we used it as our primary model for further optimization.

CatBoost Regressor

In [ ]:
!pip install catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.0 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
cat_model = CatBoostRegressor(
    depth=6,
    learning_rate=0.05,
    iterations=500,
    verbose=False,
    random_seed=42
)

cat_pipe = Pipeline([
    ('preprocess', preprocess),
    ('reg', cat_model)
])

cat_pipe.fit(x_train_r, y_train_r)
y_pred_cat = cat_pipe.predict(x_test_r)

print("===== CatBoost Regression =====")
print("MAE:", mean_absolute_error(y_test_r, y_pred_cat))
print("RMSE:", np.sqrt(mean_squared_error(y_test_r, y_pred_cat)))
print("R²:", r2_score(y_test_r, y_pred_cat))

===== CatBoost Regression =====
MAE: 3.733055295617445
RMSE: 7.235663602784353
R²: 0.09734542080921615


MAE and RMSE are higher than XGBoost, meaning CatBoost makes larger prediction errors.

R² is lower, so it explains less variance in delay outcomes.

Conclusion:
CatBoost performs reasonably well, but **it is weaker than the XGBoost baseline model on your dataset.

Grid search


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# ---------------------------
# 1) Base model using class-weight regression
# ---------------------------
xgb_reg = XGBRegressor(
    eval_metric="rmse",
    tree_method="hist",
    random_state=42
)

# ---------------------------
# 2) Grid search space (clean + efficient)
# ---------------------------
param_grid = {
    'regressor__n_estimators': [300, 500, 700],
    'regressor__learning_rate': [0.01, 0.03, 0.05],
    'regressor__max_depth': [5, 6, 7],
    'regressor__min_child_weight': [1, 3],
    'regressor__subsample': [0.7, 1.0],
    'regressor__colsample_bytree': [0.7, 0.9]
}

# ---------------------------
# 3) Pipeline
# ---------------------------
pipeline_reg = Pipeline([
    ('preprocess', preprocess),
    ('regressor', xgb_reg)
])

# ---------------------------
# 4) Grid Search (fast but thorough)
# ---------------------------
grid = GridSearchCV(
    estimator=pipeline_reg,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=2,
    n_jobs=-1
)

print("Running grid search…")
grid.fit(x_train_r, y_train_r)

# ---------------------------
# 5) Best model

# ---------------------------
# 6) Evaluate on test set
# ---------------------------

mae = mean_absolute_error(y_test_r, y_pred_grid)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_grid))
r2 = r2_score(y_test_r, y_pred_grid)

print("\n===== GRID SEARCH XGBOOST REGRESSION RESULTS =====")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

# Show examples
results_df = pd.DataFrame({
    "Actual Delay": y_test_r.values[:10],
    "Predicted Delay": y_pred_grid[:10]
})
print("\nFirst 10 predictions:\n")
print(results_df)


Running grid search…
Fitting 3 folds for each of 216 candidates, totalling 648 fits


KeyboardInterrupt: 

RandomSearch

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

# Fast search space
param_dist_fast = {
    "n_estimators": [200, 300, 400],
    "learning_rate": [0.01, 0.03, 0.05],
    "max_depth": [4, 5, 6],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 1.0],
    "min_child_weight": [1, 3],
}

xgb_reg_fast = XGBRegressor(
    eval_metric="rmse",
    tree_method="hist",
    random_state=42
)

pipeline_tune_fast = Pipeline([
    ('preprocess', preprocess),
    ('regressor', xgb_reg_fast)
])

random_search_fast = RandomizedSearchCV(
    estimator=pipeline_tune_fast,
    param_distributions={'regressor__' + k: v for k, v in param_dist_fast.items()},
    n_iter=5,                # << VERY FAST
    scoring='neg_root_mean_squared_error',
    cv=2,                    # << faster
    verbose=2,
    n_jobs=-1,
    random_state=42
)

print("Running VERY FAST tuning...")
random_search_fast.fit(x_train_r, y_train_r)

#best_model = random_search_fast.best_estimator_

# Evaluate tuned model
#y_pred_tuned = best_model.predict(x_test_r)
mae_tuned = mean_absolute_error(y_test_r, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test_r, y_pred_tuned))
r2_tuned = r2_score(y_test_r, y_pred_tuned)

print("\n===== FAST TUNED XGBOOST REGRESSION =====")
print("Best params:", random_search_fast.best_params_)
print(f"MAE:  {mae_tuned:.4f}")
print(f"RMSE: {rmse_tuned:.4f}")
print(f"R²:   {r2_tuned:.4f}")


Running VERY FAST tuning...
Fitting 2 folds for each of 5 candidates, totalling 10 fits

===== FAST TUNED XGBOOST REGRESSION =====
Best params: {'regressor__subsample': 0.7, 'regressor__n_estimators': 400, 'regressor__min_child_weight': 1, 'regressor__max_depth': 5, 'regressor__learning_rate': 0.05, 'regressor__colsample_bytree': 0.7}
MAE:  3.7026
RMSE: 7.2137
R²:   0.1028


We performed a fast hyperparameter tuning using a small randomized search (5 candidates, 2-fold CV). This method is quick but less thorough, which explains why the tuned model did not outperform the baseline XGBoost.

Optimized  RandomSearch

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

# -------------------------
# 1) Base model
# -------------------------
xgb_reg = XGBRegressor(
    eval_metric="rmse",
    tree_method="hist",
    random_state=42
)

# -------------------------
# 2) Improved Random Search Space
# -------------------------
param_dist = {
    "n_estimators": [400, 500, 600, 700, 800, 900],
    "learning_rate": [0.005, 0.01, 0.02, 0.03, 0.05],
    "max_depth": [4, 5, 6, 7],
    "min_child_weight": [1, 2, 3, 5],
    "subsample": [0.6, 0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9],
    "gamma": [0, 0.1, 0.2, 0.3],
    "reg_lambda": [0.5, 1, 2, 3],
    "reg_alpha": [0, 0.05, 0.1, 0.2]
}

# -------------------------
# 3) Pipeline
# -------------------------
pipeline_reg = Pipeline([
    ('preprocess', preprocess),
    ('regressor', xgb_reg)
])

# -------------------------
# 4) Randomized Search (improved)
# -------------------------
random_search = RandomizedSearchCV(
    estimator=pipeline_reg,
    param_distributions={'regressor__' + k: v for k, v in param_dist.items()},
    n_iter=20,          # faster and enough
    scoring='neg_root_mean_squared_error',
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

print("Running improved Random Search…")
random_search.fit(x_train_r, y_train_r)

best_model = random_search.best_estimator_

# -------------------------
# 5) Evaluate
# -------------------------
y_pred_best = best_model.predict(x_test_r)

mae = mean_absolute_error(y_test_r, y_pred_best)
rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_best))
r2 = r2_score(y_test_r, y_pred_best)

print("\n===== IMPROVED RANDOM SEARCH RESULTS =====")
print("Best params:", random_search.best_params_)
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

# Show sample predictions
results_df = pd.DataFrame({
    "Actual Delay": y_test_r.values[:10],
    "Predicted Delay": y_pred_best[:10]
})

print("\nFirst 10 predictions:\n")
print(results_df)


Running improved Random Search…
Fitting 3 folds for each of 20 candidates, totalling 60 fits

===== IMPROVED RANDOM SEARCH RESULTS =====
Best params: {'regressor__subsample': 0.8, 'regressor__reg_lambda': 1, 'regressor__reg_alpha': 0.1, 'regressor__n_estimators': 700, 'regressor__min_child_weight': 2, 'regressor__max_depth': 7, 'regressor__learning_rate': 0.05, 'regressor__gamma': 0.3, 'regressor__colsample_bytree': 0.6}
MAE:  3.5853
RMSE: 7.0900
R²:   0.1333

First 10 predictions:

   Actual Delay  Predicted Delay
0           0.0         1.383879
1          15.0         4.263115
2           0.0         3.984149
3           0.0         2.129098
4           0.0         0.428415
5           0.0         5.647308
6           0.0         6.109767
7           5.0         6.103489
8           0.0        -0.163972
9           4.0         4.969763


In [ ]:
import joblib

# Save best model to PKL
joblib.dump(best_model, "xgb_best_model.pkl")

print("Model saved as xgb_best_model.pkl")


Model saved as xgb_best_model.pkl


After expanding the hyperparameter search space and increasing cross-validation folds, the improved Random Search discovered a significantly stronger XGBoost configuration. The tuned model achieved lower MAE/RMSE and higher R² compared to the baseline, indicating better predictive accuracy and improved ability to model real flight-delay patterns.

Feature Engineering

In [ ]:
# MODEL AFTER FEATURE ENGINEERING
feature_model = Pipeline([
    ('preprocess', preprocess),
    ('regressor', XGBRegressor(
        eval_metric="rmse",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        random_state=42
    ))
])

feature_model.fit(x_train_r, y_train_r)

feat_pred = feature_model.predict(x_test_r)

feat_mae  = mean_absolute_error(y_test_r, feat_pred)
feat_rmse = np.sqrt(mean_squared_error(y_test_r, feat_pred))
feat_r2   = r2_score(y_test_r, feat_pred)

print("\n===== AFTER FEATURE ENGINEERING RESULTS =====")
print("MAE:", feat_mae)
print("RMSE:", feat_rmse)
print("R²:", feat_r2)



===== AFTER FEATURE ENGINEERING RESULTS =====
MAE: 3.7291771527614963
RMSE: 7.2517476072484355
R²: 0.09332797685622995


After adding engineered time- and ratio-based features, model performance did not improve. MAE and RMSE slightly increased and R² dropped, indicating the new features did not provide additional predictive value and may have introduced noise. The baseline XGBoost remains the more reliable configuration.